# Leaf vs Not Leaf Model Training

Upload `kaggle.json`, download Kaggle datasets, train a binary model, then download `leaf_not_leaf_model.keras` and labels.

In [ ]:
!pip install -q kaggle

from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle --version

In [ ]:
!mkdir -p /content/kaggle_data/leaf
!mkdir -p /content/kaggle_data/not_leaf

!kaggle datasets download -d csafrit2/plant-leaves-for-image-classification -p /content/kaggle_data/leaf --unzip
!kaggle datasets download -d prasunroy/natural-images -p /content/kaggle_data/not_leaf --unzip

In [ ]:
import os
import shutil
import random
from pathlib import Path

SOURCE_LEAF = Path('/content/kaggle_data/leaf')
SOURCE_NOT_LEAF = Path('/content/kaggle_data/not_leaf')

OUT_DIR = Path('/content/leaf_not_leaf_dataset')
LEAF_OUT = OUT_DIR / 'leaf'
NOT_LEAF_OUT = OUT_DIR / 'not_leaf'

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

LEAF_OUT.mkdir(parents=True, exist_ok=True)
NOT_LEAF_OUT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def collect_images(root):
    return [
        p for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

leaf_images = collect_images(SOURCE_LEAF)
not_leaf_images = collect_images(SOURCE_NOT_LEAF)

print('Leaf images found:', len(leaf_images))
print('Not leaf images found:', len(not_leaf_images))

random.seed(42)
max_count = min(len(leaf_images), len(not_leaf_images))
leaf_images = random.sample(leaf_images, max_count)
not_leaf_images = random.sample(not_leaf_images, max_count)

def copy_images(images, target_dir, prefix):
    for i, src in enumerate(images):
        dst = target_dir / f'{prefix}_{i:05d}{src.suffix.lower()}'
        shutil.copy2(src, dst)

copy_images(leaf_images, LEAF_OUT, 'leaf')
copy_images(not_leaf_images, NOT_LEAF_OUT, 'not_leaf')

print('Final leaf:', len(list(LEAF_OUT.iterdir())))
print('Final not_leaf:', len(list(NOT_LEAF_OUT.iterdir())))
print('Dataset ready:', OUT_DIR)

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    OUT_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    OUT_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

class_names = train_ds.class_names
print('Classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.12),
    layers.RandomContrast(0.12),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(2, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f'Validation accuracy: {acc * 100:.2f}%')

In [ ]:
MODEL_PATH = '/content/leaf_not_leaf_model.keras'
LABELS_PATH = '/content/leaf_not_leaf_labels.txt'

model.save(MODEL_PATH)

with open(LABELS_PATH, 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print('Saved:', MODEL_PATH)
print('Saved:', LABELS_PATH)

In [ ]:
from google.colab import files

files.download(MODEL_PATH)
files.download(LABELS_PATH)

In [ ]:
from google.colab import files
from PIL import Image
import numpy as np

uploaded = files.upload()
img_path = list(uploaded.keys())[0]

img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
arr = np.array(img)
arr = np.expand_dims(arr, axis=0)

pred = model.predict(arr)[0]
idx = int(np.argmax(pred))
label = class_names[idx]
confidence = float(pred[idx])

print('Prediction:', label)
print('Confidence:', round(confidence * 100, 2), '%')

if label == 'not_leaf' or confidence < 0.75:
    print('Result: This image looks like not_leaf, or the model is not confident enough.')
else:
    print('Result: This looks like a leaf. Send it to the disease model.')